# Import Module

In [ ]:
import ast
import hashlib
import heapq
import os
import re
from collections import Counter, defaultdict

import networkx as nx
import numpy as np

# Global Static Variabel

In [ ]:
def py_to_ast(py_code):
    try:
        return ast.parse(py_code)
    except SyntaxError as error:
        raise ValueError(f"Syntax error in the provided Python code: {error}") from error

def parse_commit_python_files(commit):
    trees = {}
    parse_errors = {}
    for item in commit.tree.traverse():
        if item.type != "blob" or not item.path.endswith(".py"):
            continue
        source = item.data_stream.read().decode("utf-8", errors="replace")
        try:
            trees[item.path] = py_to_ast(source)
        except ValueError as error:
            parse_errors[item.path] = str(error)
    return trees, parse_errors

In [ ]:
LEAF = (ast.expr_context, ast.operator, ast.boolop, ast.unaryop, ast.cmpop)
FUNC = (ast.FunctionDef, ast.AsyncFunctionDef)
SCOPES = FUNC + (ast.ClassDef, ast.Lambda, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

BINDERS = {getattr(ast, n): f for n, f in [("ExceptHandler", "name"), ("MatchAs", "name"), ("MatchStar", "name"), ("MatchMapping", "rest")] if hasattr(ast, n)}

CONTROL = {getattr(ast, n): (c, b) for n, c, b in [
    ("If",            ("test",),            ("body", "orelse")),
    ("While",         ("test",),            ("body", "orelse")),
    ("For",           ("iter",),            ("body", "orelse")),
    ("AsyncFor",      ("iter",),            ("body", "orelse")),
    ("With",          ("items",),           ("body",)),
    ("AsyncWith",     ("items",),           ("body",)),
    ("Try",           (),                   ("body", "handlers", "orelse", "finalbody")),
    ("TryStar",       (),                   ("body", "handlers", "orelse", "finalbody")),
    ("ExceptHandler", ("type",),            ("body",)),
    ("Match",         ("subject",),         ("cases",)),
    ("match_case",    ("pattern", "guard"), ("body",)),
] if hasattr(ast, n)}

EDGE_PRIORITY = {
    "call": 0,
    "data_dep": 1,
    "import": 2,
    "control": 3,
    "arg_bind": 4,
    "return_flow": 5,
    "class_method": 6,
    "child": 7,
}

NODE_TYPES = tuple(name for name in dir(ast) if isinstance(getattr(ast, name), type) and issubclass(getattr(ast, name), ast.AST))
NODE_TYPE_TO_ID = {name: index for index, name in enumerate(NODE_TYPES)}

EDGE_TYPES = (
    "child",
    "call",
    "data_dep",
    "import",
    "control",
    "arg_bind",
    "return_flow",
    "class_method",
)

EDGE_TYPE_TO_ID = {name: index for index, name in enumerate(EDGE_TYPES)}

AST_TYPES = tuple(name for name in sorted(dir(ast)) if isinstance(getattr(ast, name), type) and issubclass(getattr(ast, name), ast.AST))
AST_TYPE_TO_ID = {name: index for index, name in enumerate(AST_TYPES)}

CHANGE_STATUS_TO_ID = {"unchanged": 0, "modified": 1, "added": 2, "deleted": 3}

TOKEN_BUCKETS = 64

# Graph Builder

In [ ]:
class Graph:
    def __init__(self):
        self.graph = nx.MultiDiGraph()

    def add_node(self, node_idx, node_type, start_lineno = None, end_lineno = None, attrs = None):
        self.graph.add_node(node_idx, type = node_type, start_lineno = start_lineno, end_lineno = end_lineno, attrs = attrs or {})

    def add_edge(self, from_node, to_node, edge_type, **attrs):
        self.graph.add_edge(from_node, to_node, type = edge_type, **attrs)

class Scope:
    def __init__(self, node, parent, path = None):
        self.node, self.parent, self.path = node, parent, path
        self.defs = defaultdict(list)

class Builder:
    def __init__(self):
        self.graph = Graph()
        self._ids = {}
        self.scope_of = {}
        self.own_scope = {}
        self._defpos = {}
        self.module_scope = {}

    @staticmethod
    def _module_name(path):
        path = path.replace("\\", "/")
        if path.endswith(".py"):
            path = path[:-3]
        parts = path.split("/")
        if parts[-1] == "__init__":
            parts = parts[:-1]
        return ".".join(parts)

    @staticmethod
    def _children(node):
        for field, value in ast.iter_fields(node):
            items = value if isinstance(value, list) else [value]
            for order, child in enumerate(items):
                if isinstance(child, ast.AST) and not isinstance(child, LEAF):
                    yield field, order, child

    @staticmethod
    def _attrs(node):
        attrs = {}
        for field, value in ast.iter_fields(node):
            if isinstance(value, LEAF):
                attrs[field] = type(value).__name__
            elif isinstance(value, list):
                if value and all(isinstance(v, LEAF) for v in value):
                    attrs[field] = [type(v).__name__ for v in value]
                elif value and all(isinstance(v, str) for v in value):
                    attrs[field] = list(value)
            elif value is not None and not isinstance(value, ast.AST):
                attrs[field] = value
        return attrs

    @staticmethod
    def _register(node, scope):
        if isinstance(node, FUNC + (ast.ClassDef,)):
            scope.defs[node.name].append(node)
        elif isinstance(node, ast.arg):
            scope.defs[node.arg].append(node)
        elif isinstance(node, ast.Name) and isinstance(node.ctx, ast.Store):
            scope.defs[node.id].append(node)
        elif isinstance(node, ast.alias):
            name = node.asname or node.name.split(".")[0]
            if name != "*":
                scope.defs[name].append(node)
        elif type(node) in BINDERS:
            name = getattr(node, BINDERS[type(node)])
            if name:
                scope.defs[name].append(node)

    @staticmethod
    def _inside(node, field):
        if isinstance(node, FUNC):
            return field in ("args", "body")
        if isinstance(node, ast.ClassDef):
            return field == "body"
        return True

    @staticmethod
    def _lookup(scope, name):
        s, start = scope, True
        while s is not None:
            if name in s.defs and (start or not isinstance(s.node, ast.ClassDef)):
                return s
            s, start = s.parent, False
        return None

    def build(self, trees):
        if isinstance(trees, ast.AST):
            trees = {"module.py": trees}

        for path, tree in trees.items():
            stack = [(tree, None, None, None)]
            while stack:
                node, parent, field, order = stack.pop()
                idx = self._ids.setdefault(node, len(self._ids))
                self.graph.add_node(idx, type(node).__name__, start_lineno = getattr(node, 'lineno', None), end_lineno = getattr(node, 'end_lineno', None), attrs = self._attrs(node))
                if parent is not None:
                    self.graph.add_edge(parent, idx, "child", field = field, order = order)
                for field, order, child in reversed(list(self._children(node))):
                    stack.append((child, idx, field, order))
            self.graph.graph.nodes[self._ids[tree]]["attrs"]["path"] = path

        all_nodes = [n for t in trees.values() for n in ast.walk(t)]

        for n in all_nodes:
            if isinstance(n, (ast.Assign, ast.AnnAssign, ast.AugAssign, ast.NamedExpr)):
                targets = n.targets if isinstance(n, ast.Assign) else [n.target]
                for t in targets:
                    for m in ast.walk(t):
                        if isinstance(m, ast.Name) and isinstance(m.ctx, ast.Store):
                            self._defpos[m] = (n.end_lineno, n.end_col_offset)

        for path, tree in trees.items():
            root = self.module_scope[path] = Scope(tree, None, path)
            stack = [(tree, root)]
            while stack:
                node, scope = stack.pop()
                self.scope_of[node] = scope
                self._register(node, scope)
                inner = scope
                if isinstance(node, SCOPES):
                    inner = self.own_scope[node] = Scope(node, scope)
                for field, _, child in self._children(node):
                    if isinstance(node, ast.arguments) and field in ("defaults", "kw_defaults"):
                        cs = scope.parent
                    elif isinstance(node, SCOPES) and not self._inside(node, field):
                        cs = scope
                    else:
                        cs = inner
                    stack.append((child, cs))

        modules = {self._module_name(p): r for p, r in self.module_scope.items()}
        info, alias_defs, alias_mods = {}, {}, {}

        def module_of(scope):
            while scope.parent is not None:
                scope = scope.parent
            return scope.path

        for imp in all_nodes:
            if isinstance(imp, (ast.Import, ast.ImportFrom)):
                for al in imp.names:
                    info[al] = (imp, module_of(self.scope_of[imp]))

        def find_module(name):
            if not name:
                return None
            if name in modules:
                return modules[name]
            cand = [m for m in modules if m.endswith("." + name)]
            return modules[cand[0]] if len(cand) == 1 else None

        def absolute_module(imp, path):
            if imp.level == 0:
                return imp.module or ""
            parts = self._module_name(path).split(".")
            if not path.endswith("__init__.py"):
                parts = parts[:-1]
            parts = parts[:max(0, len(parts) - (imp.level - 1))]
            return ".".join(parts + (imp.module.split(".") if imp.module else []))

        def resolve_alias(al, seen=()):
            if al in alias_defs:
                return
            imp, path = info[al]
            defs, mods = [], []
            if isinstance(imp, ast.Import):
                m = find_module(al.name if al.asname else al.name.split(".")[0])
                if m:
                    mods.append(m)
            elif al.name != "*":
                base = absolute_module(imp, path)
                m = find_module(base)
                for d in (m.defs.get(al.name, []) if m else []):
                    if isinstance(d, ast.alias) and d in info and d is not al and d not in seen:
                        resolve_alias(d, seen + (al,))
                        defs += alias_defs.get(d, [])
                        mods += alias_mods.get(d, [])
                    else:
                        defs.append(d)
                if not defs and not mods:
                    sub = find_module(f"{base}.{al.name}" if base else al.name)
                    if sub:
                        mods.append(sub)
            alias_defs[al], alias_mods[al] = defs, mods

        for al in info:
            resolve_alias(al)
        for al in info:
            for d in alias_defs[al]:
                self.graph.add_edge(self._ids[al], self._ids[d], "import")
            for m in alias_mods[al]:
                self.graph.add_edge(self._ids[al], self._ids[m.node], "import")

        def expand(ds):
            out = []
            for d in ds:
                out += alias_defs[d] if isinstance(d, ast.alias) else [d]
            return out

        def defs_of(scope, name, raw=False):
            s = self._lookup(scope, name)
            ds = s.defs[name] if s else []
            return ds if raw else expand(ds)

        def class_scopes(scope, name):
            s = scope
            while s is not None:
                if isinstance(s.node, FUNC) and s.parent and isinstance(s.parent.node, ast.ClassDef):
                    a = s.node.args
                    params = a.posonlyargs + a.args
                    if params and params[0].arg == name:
                        return [s.parent], True
                    break
                s = s.parent
            out = [self.own_scope[d] for d in defs_of(scope, name) if isinstance(d, ast.ClassDef)]
            for d in defs_of(scope, name, raw=True):
                if isinstance(d, ast.alias):
                    out += alias_mods[d]
            return out, False

        def pos(d):
            return self._defpos.get(d) or (getattr(d, "lineno", 0), getattr(d, "col_offset", 0))

        def add_target(found, d, offset):
            if isinstance(d, FUNC):
                found.append((d, offset))
            elif isinstance(d, ast.ClassDef):
                found.append((d, 0))
                for m in self.own_scope[d].defs.get("__init__", []):
                    if isinstance(m, FUNC):
                        found.append((m, 1))

        for cls in all_nodes:
            if isinstance(cls, ast.ClassDef):
                for stmt in cls.body:
                    if isinstance(stmt, FUNC):
                        self.graph.add_edge(self._ids[cls], self._ids[stmt], "class_method")

        calls = []
        for call in all_nodes:
            if not isinstance(call, ast.Call):
                continue
            f, scope = call.func, self.scope_of[call]
            found = []
            if isinstance(f, ast.Name):
                for d in defs_of(scope, f.id):
                    add_target(found, d, 0)
            elif isinstance(f, ast.Attribute) and isinstance(f.value, ast.Name):
                scopes, bound = class_scopes(scope, f.value.id)
                for cs in scopes:
                    for d in expand(cs.defs.get(f.attr, [])):
                        if isinstance(d, FUNC) and isinstance(cs.node, ast.ClassDef):
                            decos = {x.id for x in d.decorator_list if isinstance(x, ast.Name)}
                            offset = 0 if "staticmethod" in decos else 1 if (bound or "classmethod" in decos) else 0
                            add_target(found, d, offset)
                        else:
                            add_target(found, d, 0)
            for t, offset in found:
                self.graph.add_edge(self._ids[call], self._ids[t], "call")
                calls.append((call, t, offset))

        callee = {c.func for c in all_nodes if isinstance(c, ast.Call)}
        aug = {n.target for n in all_nodes if isinstance(n, ast.AugAssign) and isinstance(n.target, ast.Name)}
        for use in all_nodes:
            if not isinstance(use, ast.Name) or use in callee:
                continue
            if not (isinstance(use.ctx, ast.Load) or use in aug):
                continue
            scope = self.scope_of[use]
            found = self._lookup(scope, use.id)
            if not found:
                continue
            defs = found.defs[use.id]
            if found is scope:
                before = [d for d in defs if pos(d) < (use.lineno, use.col_offset)]
                defs = before or defs
            for d in defs:
                if d is not use:
                    self.graph.add_edge(self._ids[d], self._ids[use], "data_dep")

        returns = {}

        def returns_of(fn):
            if fn not in returns:
                out, st = [], list(fn.body)
                while st:
                    n = st.pop()
                    if isinstance(n, ast.Return):
                        out.append(n)
                    if not isinstance(n, FUNC + (ast.ClassDef, ast.Lambda)):
                        st.extend(ast.iter_child_nodes(n))
                returns[fn] = out
            return returns[fn]

        for call, t, offset in calls:
            if isinstance(t, ast.ClassDef):
                continue
            a = t.args
            params = a.posonlyargs + a.args
            by_name = {p.arg: p for p in params + a.kwonlyargs}
            for i, arg in enumerate(call.args):
                if not isinstance(arg, ast.Starred) and i + offset < len(params):
                    self.graph.add_edge(self._ids[arg], self._ids[params[i + offset]], "arg_bind")
            for kw in call.keywords:
                if kw.arg in by_name:
                    self.graph.add_edge(self._ids[kw.value], self._ids[by_name[kw.arg]], "arg_bind")
            for r in returns_of(t):
                if r.value is not None:
                    self.graph.add_edge(self._ids[r], self._ids[call], "return_flow")

        for n in all_nodes:
            rule = CONTROL.get(type(n))
            if rule is None:
                continue
            ctrl_fields, block_fields = rule
            sources = [c for f, _, c in self._children(n) if f in ctrl_fields] or [n]
            for f, _, child in self._children(n):
                if f in block_fields:
                    for s in sources:
                        self.graph.add_edge(self._ids[s], self._ids[child], "control", field = f)

        return self.graph

# Graph Pruned System

In [ ]:
def _range_overlaps(attrs, ranges):
    start = attrs.get("start_lineno")
    end = attrs.get("end_lineno")
    return (
        start is not None
        and end is not None
        and any(start <= right and left <= end for left, right in ranges)
    )

def prepare_pruned_graph(builder, trees, changed_ranges, max_nodes):
    graph = builder.graph.graph
    metadata = {}
    anchors = set()
    for path, tree in trees.items():
        for node in ast.walk(tree):
            if node not in builder._ids:
                continue
            node_id = builder._ids[node]
            info = {
                "path": path,
                "ast_node": node,
                "start_lineno": getattr(node, "lineno", None),
                "end_lineno": getattr(node, "end_lineno", None),
            }
            metadata[node_id] = info
            if _range_overlaps(
                info,
                changed_ranges.get(path, ()),
            ):
                anchors.add(node_id)
    if not anchors:
        raise ValueError("Tidak ditemukan node AST pada changed_ranges")
    if len(anchors) > max_nodes:
        raise ValueError("Jumlah change anchor melebihi max_nodes")

    priority = {
        "call": 0,
        "data_dep": 1,
        "import": 2,
        "control": 3,
        "arg_bind": 4,
        "return_flow": 5,
        "class_method": 6,
        "child": 7,
    }
    best = {node_id: (0, -1, node_id) for node_id in anchors}
    queue = list(best.values())
    heapq.heapify(queue)
    while queue:
        distance, rank, node_id = heapq.heappop(queue)
        if best.get(node_id) != (distance, rank, node_id):
            continue
        neighbors = [
            (source, attrs.get("type"))
            for source, _, attrs in graph.in_edges(node_id, data=True)
        ]
        neighbors.extend(
            (target, attrs.get("type"))
            for _, target, attrs in graph.out_edges(node_id, data=True)
        )
        for neighbor, edge_type in neighbors:
            candidate = (
                distance + 1,
                priority.get(edge_type, len(priority)),
                neighbor,
            )
            if neighbor not in best or candidate < best[neighbor]:
                best[neighbor] = candidate
                heapq.heappush(queue, candidate)
    selected = sorted(best, key=best.get)[:max_nodes]
    result = graph.subgraph(selected).copy()
    result = nx.convert_node_labels_to_integers(
        result, first_label=0, ordering="sorted", label_attribute="original_id"
    )
    result.graph["max_nodes"] = max_nodes
    result.graph["anchor_original_ids"] = sorted(anchors)
    anchors = set(result.graph.get("anchor_original_ids", ()))
    for node_id, attrs in result.nodes(data=True):
        original_id = attrs.get("original_id", node_id)
        info = metadata.get(original_id, {})
        source_attrs = dict(attrs)
        source_attrs["path"] = info.get("path")
        source_attrs["start_lineno"] = info.get("start_lineno")
        source_attrs["end_lineno"] = info.get("end_lineno")
        source_attrs["ast_node"] = info.get("ast_node")
        source_attrs["is_change_anchor"] = int(original_id in anchors)
        source_attrs["is_changed_file"] = int(
            info.get("path") in changed_ranges
        )
        result.nodes[node_id].update(source_attrs)
    return result

# Git Diff Metadata

In [ ]:
def diff_metadata(diff_items):
    result = {
        "old_ranges": {},
        "new_ranges": {},
        "edits": [],
        "changed_files": [],
    }
    header = re.compile(r"^@@ -(\d+)(?:,(\d+))? \+(\d+)(?:,(\d+))? @@")
    for item in diff_items:
        old_path = item.a_path
        new_path = item.b_path
        path = new_path or old_path
        if not path or not path.endswith(".py"):
            continue
        result["changed_files"].append(path)
        old_ranges, new_ranges = [], []
        old_line = new_line = None
        removed, added = [], []
        removed_start = added_start = None

        def flush():
            nonlocal removed, added, removed_start, added_start
            if removed or added:
                if removed and added:
                    operation = "Update"
                elif removed:
                    operation = "Delete"
                else:
                    operation = "Insert"
                result["edits"].append(
                    {
                        "operation": operation,
                        "path": path,
                        "old_start": removed_start,
                        "new_start": added_start,
                        "old_text": "\n".join(removed),
                        "new_text": "\n".join(added),
                    }
                )
            removed = []
            added = []
            removed_start = added_start = None

        for line in item.diff.decode("utf-8", errors = "replace").splitlines():
            match = header.match(line)
            if match:
                flush()
                old_line = int(match.group(1))
                new_line = int(match.group(3))
                continue
            if old_line is None or line.startswith("\\"):
                continue
            if line.startswith("-") and not line.startswith("---"):
                if removed_start is None:
                    removed_start = old_line
                removed.append(line[1:])
                old_ranges.append((old_line, old_line))
                old_line += 1
            elif line.startswith("+") and not line.startswith("+++"):
                if added_start is None:
                    added_start = new_line
                added.append(line[1:])
                new_ranges.append((new_line, new_line))
                new_line += 1
            else:
                flush()
                old_line += 1
                new_line += 1
        flush()
        if old_ranges:
            result["old_ranges"][old_path or path] = old_ranges
        if new_ranges:
            result["new_ranges"][new_path or path] = new_ranges

    for key in ("old_ranges", "new_ranges"):
        for path, ranges in result[key].items():
            ranges.sort()
            merged = []
            for start, end in ranges:
                if merged and start <= merged[-1][1] + 1:
                    merged[-1] = (merged[-1][0], max(merged[-1][1], end))
                else:
                    merged.append((start, end))
            result[key][path] = merged
    return result

# Feature

In [ ]:
def pair_features(diff_info):
    edits = diff_info["edits"]
    operation_ids = {"Insert": 1, "Delete": 2, "Update": 3}
    counts = Counter(edit["operation"] for edit in edits)
    return np.asarray(
        [
            len(diff_info["changed_files"]),
            len(edits),
            counts.get("Insert", 0),
            counts.get("Delete", 0),
            counts.get("Update", 0),
            sum(len(edit["old_text"].split()) for edit in edits),
            sum(len(edit["new_text"].split()) for edit in edits),
            sum(operation_ids[edit["operation"]] for edit in edits),
        ],
        dtype=np.float32,
    )

# Graph to Tensor

In [ ]:
def _stable_hash(value, buckets = 64):
    digest = hashlib.blake2b(str(value).encode("utf-8"), digest_size=8).digest()
    return int.from_bytes(digest, "big") % buckets

def _node_tokens(attrs):
    values = [attrs.get("type", "")]
    ast_attrs = attrs.get("attrs", {})
    for key in sorted(ast_attrs):
        value = ast_attrs[key]
        if isinstance(value, (str, int, float, bool)):
            values.append(f"{key}={value}")
        elif isinstance(value, list):
            values.extend(f"{key}={item}" for item in value)
    return values

class Tensor:
    def __init__(
        self,
        pruned_graph,
        graph_features=None,
        local_change_features=None,
    ):
        self.graph = pruned_graph
        self.graph_features = self._features(graph_features)
        self.local_change_features = self._features(local_change_features)
        self.node_ids = list(pruned_graph.nodes())
        self.node_to_index = {node: i for i, node in enumerate(self.node_ids)}
        self.x = self._node_features()
        self.edge_index, self.edge_type = self._edges()

    @staticmethod
    def _features(values):
        if values is None:
            return np.empty(0, dtype=np.float32)
        if isinstance(values, dict):
            values = [values[key] for key in sorted(values)]
        vector = np.asarray(values, dtype=np.float32).reshape(-1)
        if not np.isfinite(vector).all():
            raise ValueError("Fitur mengandung NaN atau infinite")
        return vector

    def _node_features(self):
        type_count = len(AST_TYPES) + 1
        width = type_count + TOKEN_BUCKETS + 7
        output = np.zeros((len(self.node_ids), width), dtype=np.float32)
        line_numbers = [
            value
            for node in self.node_ids
            for value in (
                self.graph.nodes[node].get("start_lineno"),
                self.graph.nodes[node].get("end_lineno"),
            )
            if isinstance(value, (int, float))
        ]
        max_line = max(line_numbers, default=1)
        for row, node in enumerate(self.node_ids):
            attrs = self.graph.nodes[node]
            type_id = AST_TYPE_TO_ID.get(attrs.get("type"), len(AST_TYPES))
            output[row, type_id] = 1.0
            offset = type_count
            for token in _node_tokens(attrs):
                output[row, offset + _stable_hash(token)] = 1.0
            start = attrs.get("start_lineno") or 0
            end = attrs.get("end_lineno") or start
            output[row, offset + TOKEN_BUCKETS] = start / max_line
            output[row, offset + TOKEN_BUCKETS + 1] = end / max_line
            output[row, offset + TOKEN_BUCKETS + 2] = attrs.get(
                "is_change_anchor", 0
            )
            output[row, offset + TOKEN_BUCKETS + 3] = attrs.get(
                "is_changed_file", 0
            )
            output[row, offset + TOKEN_BUCKETS + 4] = int(
                bool(attrs.get("attrs", {}).get("ctx"))
            )
            output[row, offset + TOKEN_BUCKETS + 5] = _stable_hash(
                attrs.get("path", ""), 16
            ) / 16
            output[row, offset + TOKEN_BUCKETS + 6] = int(
                attrs.get("ast_node") is not None
            )
        return output

    def _edges(self):
        edges, types = [], []
        for source, target, attrs in self.graph.edges(data=True):
            edge_type = attrs.get("type")
            if edge_type not in EDGE_TYPE_TO_ID:
                raise ValueError(f"Tipe edge tidak dikenal: {edge_type!r}")
            edges.append(
                [self.node_to_index[source], self.node_to_index[target]]
            )
            types.append(EDGE_TYPE_TO_ID[edge_type])
        index = np.asarray(edges, dtype=np.int64).T if edges else np.empty((2, 0), dtype=np.int64)
        return index, np.asarray(types, dtype=np.int64)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import RGCNConv, global_add_pool, global_mean_pool, global_max_pool

class RGCNEncoder(nn.Module):
    def __init__(self, in_channels, hidden_channels, num_relations = len(EDGE_TYPES), num_layers=2, readout="add"):
        super().__init__()
        self.convs = nn.ModuleList(
            RGCNConv(in_channels if i == 0 else hidden_channels, hidden_channels, num_relations) for i in range(num_layers)
        )
        self.readout = {"add": global_add_pool, "average": global_mean_pool, "max": global_max_pool}[readout]

    def forward(self, x, edge_index, edge_type, batch = None):
        if batch is None:  # satu graph saja
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index, edge_type)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return self.readout(x, batch)

class PairRGCN(nn.Module):
    def __init__(self, in_channels, hidden_channels, graph_feat_dim, num_classes=2, dropout=0.1, readout="add"):
        super().__init__()
        self.encoder = RGCNEncoder(in_channels, hidden_channels, readout=readout)
        self.head = nn.Sequential(
            nn.Linear(hidden_channels * 3 + graph_feat_dim, hidden_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, num_classes),
        )

    def forward(self, parent, current, graph_feat):
        hp = self.encoder(parent.x, parent.edge_index, parent.edge_type, getattr(parent, "batch", None))
        hc = self.encoder(current.x, current.edge_index, current.edge_type, getattr(current, "batch", None))
        return self.head(torch.cat([hp, hc, hc - hp, graph_feat], dim=-1))

def to_data(t):
    return Data(
        x=torch.from_numpy(t.x),
        edge_index=torch.from_numpy(t.edge_index),
        edge_type=torch.from_numpy(t.edge_type),
    )

# Visualization

In [ ]:
from collections.abc import Iterable

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


EDGE_STYLES = {
    "child": {"color": "#7f8c8d", "style": "solid", "width": 1.0},
    "call": {"color": "#8e44ad", "style": "solid", "width": 1.8},
    "data_dep": {"color": "#c0392b", "style": "dashed", "width": 1.4},
    "import": {"color": "#d35400", "style": "solid", "width": 1.4},
    "control": {"color": "#2980b9", "style": "dashed", "width": 1.2},
    "arg_bind": {"color": "#16a085", "style": "dotted", "width": 1.2},
    "return_flow": {"color": "#27ae60", "style": "dashed", "width": 1.4},
    "class_method": {"color": "#2c3e50", "style": "solid", "width": 1.8},
}

NODE_COLORS = {
    "Module": "#95a5a6",
    "ClassDef": "#82b366",
    "FunctionDef": "#6fa8dc",
    "AsyncFunctionDef": "#6fa8dc",
    "Call": "#c27ba0",
    "Name": "#ffd966",
    "Assign": "#e06666",
    "Import": "#f6b26b",
    "ImportFrom": "#f6b26b",
}
DEFAULT_NODE_COLOR = "#d9d9d9"
CHANGE_ANCHOR_COLOR = "#fff2cc"
CHANGE_ANCHOR_EDGE_COLOR = "#e74c3c"


def _networkx_graph(graph):
    """Return the NetworkX graph stored by the project's Graph class."""
    if isinstance(graph, (nx.Graph, nx.DiGraph, nx.MultiGraph, nx.MultiDiGraph)):
        return graph
    network = getattr(graph, "graph", None)
    if not isinstance(network, (nx.Graph, nx.DiGraph, nx.MultiGraph, nx.MultiDiGraph)):
        raise TypeError("graph harus merupakan instance Graph dengan atribut graph NetworkX")
    return network


def _node_label(node_id, attributes):
    node_type = attributes.get("type", "Node")
    node_attrs = attributes.get("attrs") or {}
    name = node_attrs.get("name") or node_attrs.get("id") or node_attrs.get("value")
    location = attributes.get("start_lineno")
    details = [str(value) for value in (name, f"baris {location}" if location else None) if value]
    return f"{node_type}\n{' | '.join(details)}" if details else f"{node_type}\n#{node_id}"


def visualize_graph(
    graph,
    *,
    figsize=(18, 12),
    layout="spring",
    edge_types: Iterable[str] | None = None,
    with_labels=True,
    node_size=900,
    font_size=10,
    seed=42,
    title="Graph visualization",
):
    """Draw a Graph object without changing the Graph class or its data.

    Returns the ``(figure, axes)`` pair so callers can further customize or
    save the visualization with ``figure.savefig(...)``.
    """
    network = _networkx_graph(graph)
    if network.number_of_nodes() == 0:
        raise ValueError("Graph kosong; tidak ada node untuk divisualisasikan")

    if layout == "dot":
        try:
            positions = nx.nx_agraph.graphviz_layout(network, prog="dot")
        except (ImportError, RuntimeError, OSError):
            positions = nx.spring_layout(network, seed=seed)
    elif layout == "kamada_kawai":
        positions = nx.kamada_kawai_layout(network)
    elif layout == "circular":
        positions = nx.circular_layout(network)
    elif layout == "spring":
        positions = nx.spring_layout(network, seed=seed)
    else:
        raise ValueError("layout harus salah satu dari: spring, dot, kamada_kawai, circular")

    active_types = list(edge_types) if edge_types is not None else sorted(
        {data.get("type") for _, _, data in network.edges(data=True)}
    )
    unknown_types = set(active_types) - set(EDGE_STYLES)
    if unknown_types:
        raise ValueError(f"Tipe edge tidak dikenal: {', '.join(sorted(unknown_types))}")

    figure, axes = plt.subplots(figsize=figsize)
    node_colors = [
        NODE_COLORS.get(network.nodes[node].get("type"), DEFAULT_NODE_COLOR)
        for node in network.nodes
    ]
    labels = {
        node: _node_label(node, network.nodes[node])
        for node in network.nodes
    }
    anchor_nodes = [
        node
        for node, attributes in network.nodes(data=True)
        if attributes.get("is_change_anchor")
    ]

    for edge_type in active_types:
        style = EDGE_STYLES[edge_type]
        edges = [
            (source, target, key)
            for source, target, key, data in network.edges(keys=True, data=True)
            if data.get("type") == edge_type
        ]
        if edges:
            nx.draw_networkx_edges(
                network,
                positions,
                ax=axes,
                edgelist=edges,
                edge_color=style["color"],
                style=style["style"],
                width=style["width"],
                arrows=network.is_directed(),
                arrowsize=12,
                connectionstyle="arc3,rad=0.05",
            )

    nx.draw_networkx_nodes(
        network,
        positions,
        ax=axes,
        node_color=node_colors,
        node_size=node_size,
        edgecolors="#333333",
        linewidths=0.7,
    )
    if anchor_nodes:
        nx.draw_networkx_nodes(
            network,
            positions,
            ax=axes,
            nodelist=anchor_nodes,
            node_color=CHANGE_ANCHOR_COLOR,
            node_size=max(node_size * 1.8, node_size + 500),
            edgecolors=CHANGE_ANCHOR_EDGE_COLOR,
            linewidths=2.8,
        )
        center = np.mean(
            np.asarray([positions[node] for node in network.nodes]),
            axis=0,
        )
        for index, node in enumerate(anchor_nodes):
            direction = np.asarray(positions[node]) - center
            length = np.linalg.norm(direction)
            if length == 0:
                direction = np.asarray([0.0, 1.0])
                length = 1.0
            offset = tuple(direction / length * 70)
            axes.annotate(
                "CHANGE ANCHOR" if index == 0 else "",
                xy=positions[node],
                xytext=offset,
                textcoords="offset points",
                ha="center",
                va="center",
                fontsize=max(font_size, 11),
                fontweight="bold",
                color=CHANGE_ANCHOR_EDGE_COLOR,
                arrowprops={
                    "arrowstyle": "-|>",
                    "color": CHANGE_ANCHOR_EDGE_COLOR,
                    "lw": 3.5,
                    "mutation_scale": 28,
                    "shrinkA": 8,
                    "shrinkB": 10,
                },
                zorder=5,
            )
    if with_labels:
        nx.draw_networkx_labels(network, positions, labels, ax=axes, font_size=font_size)

    node_legend = [
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            label=node_type,
            markerfacecolor=color,
            markeredgecolor="#333333",
            markersize=8,
        )
        for node_type, color in NODE_COLORS.items()
        if node_type in {data.get("type") for _, data in network.nodes(data=True)}
    ]
    edge_legend = [
        Line2D(
            [0],
            [0],
            color=EDGE_STYLES[edge_type]["color"],
            linestyle=EDGE_STYLES[edge_type]["style"],
            linewidth=2,
            label=edge_type,
        )
        for edge_type in active_types
    ]
    anchor_legend = [
        Line2D(
            [0],
            [0],
            marker="o",
            color=CHANGE_ANCHOR_EDGE_COLOR,
            label="change anchor",
            markerfacecolor=CHANGE_ANCHOR_COLOR,
            markeredgecolor=CHANGE_ANCHOR_EDGE_COLOR,
            markersize=10,
            linewidth=2,
        )
    ] if anchor_nodes else []
    axes.legend(handles=node_legend + anchor_legend + edge_legend, loc="upper left", bbox_to_anchor=(1.02, 1))
    axes.set_title(title)
    axes.axis("off")
    figure.tight_layout()
    return figure, axes


# Training with Defectors Dataset

In [ ]:
REPO_PATH = r"repositories/"
MAX_NODES = 1000000

In [ ]:
import pandas as pd

df_train = pd.read_parquet(r"datasets\Defectors\line_bug_prediction_splits\random\train.parquet.gzip", engine="pyarrow")
df_train["is_bug"] = df_train["lines"].apply(lambda x: len(x) > 0)
class_counts_train = df_train["is_bug"].value_counts()

print("===Train Data===")
print(f"Total Data : {len(df_train)}")
print(f"Non-Bug (False): {class_counts_train.get(False, 0)}")
print(f"Bug (True): {class_counts_train.get(True, 0)}")

df_val = pd.read_parquet(r"datasets\Defectors\line_bug_prediction_splits\random\val.parquet.gzip", engine="pyarrow")
df_val["is_bug"] = df_val["lines"].apply(lambda x: len(x) > 0)
class_counts_val = df_val["is_bug"].value_counts()

print("===Val Data===")
print(f"Total Data : {len(df_val)}")
print(f"Non-Bug (False): {class_counts_val.get(False, 0)}")
print(f"Bug (True): {class_counts_val.get(True, 0)}")

df_test = pd.read_parquet(r"datasets\Defectors\line_bug_prediction_splits\random\test.parquet.gzip", engine="pyarrow")
df_test["is_bug"] = df_test["lines"].apply(lambda x: len(x) > 0)
class_counts_test = df_test["is_bug"].value_counts()

print("===Test Data===")
print(f"Total Data : {len(df_test)}")
print(f"Non-Bug (False): {class_counts_test.get(False, 0)}")
print(f"Bug (True): {class_counts_test.get(True, 0)}")

In [ ]:
from git import Repo

training_data = []

for i, row in df_train.iterrows():
    project_name = row["repo"]
    current_commit_hash = row["commit"]

    repo_path = os.path.join(REPO_PATH, project_name)    

    try:
        if os.path.exists(repo_path):
            print(f"Repository already exists: {repo_path}")
            repo = Repo(repo_path)
        else:
            # os.mkdir(repo_path)
            # print(f"Cloning repository: {url} to {repo_path}")
            # repo = Repository(url, clone_repo_to = REPO_PATH)
            continue
    except Exception as e:
        print(f"Error occurred while preparing repositories: {e}")
        continue

    current_commit = repo.commit(current_commit_hash)
    parent_commit = repo.head.commit

    if current_commit.hexsha != str(current_commit_hash):
        print(f"commit_sha tidak cocok dengan commit yang dibaca. Skipping...")
        continue

    trees_parent, parent_parse_errors = parse_commit_python_files(parent_commit)
    trees_current, current_parse_errors = parse_commit_python_files(current_commit)

    builder_parent = Builder()
    graph_parent = builder_parent.build(trees_parent)
    
    builder_current = Builder()
    graph_current = builder_current.build(trees_current)

    diff = parent_commit.diff(current_commit, create_patch = True)
    diff_info = diff_metadata(diff)
    if not diff_info["changed_files"]:
        raise ValueError("Tidak ada perubahan file Python pada commit pair ini")

    try:
        parent_graph = prepare_pruned_graph(
            builder_parent, trees_parent, diff_info["old_ranges"], max_nodes=MAX_NODES
        )
        current_graph = prepare_pruned_graph(
            builder_current, trees_current, diff_info["new_ranges"], max_nodes=MAX_NODES
        )

    except Exception as e:
        print(f"Error occurred while preparing pruned graphs: {e}")
        continue

    change_features = pair_features(diff_info)

    parent_input_tensor = Tensor(
        parent_graph,
        graph_features=change_features,
        local_change_features=change_features
    )

    current_input_tensor = Tensor(
        current_graph,
        graph_features=change_features,
        local_change_features=change_features
    )

    training_data.append({
        "parent": to_data(parent_input_tensor),
        "current": to_data(current_input_tensor),
        "features": torch.from_numpy(change_features).unsqueeze(0),
        "y": torch.tensor([1 if row["is_bug"] else 0], dtype=torch.long),
        "project": project_name,
    })

print(f"Prepared training data for {len(training_data)} commit pairs.")

In [ ]:
val_data = []

for i, row in df_val.iterrows():
    project_name = row["repo"]
    current_commit_hash = row["commit"]

    repo_path = os.path.join(REPO_PATH, project_name)    

    repo = Repo(repo_path)
    current_commit = repo.commit(current_commit_hash)
    parent_commit = repo.head.commit

    if current_commit.hexsha != str(current_commit_hash):
        print(f"commit_sha tidak cocok dengan commit yang dibaca. Skipping...")
        continue

    trees_parent, parent_parse_errors = parse_commit_python_files(parent_commit)
    trees_current, current_parse_errors = parse_commit_python_files(current_commit)

    builder_parent = Builder()
    graph_parent = builder_parent.build(trees_parent)
    
    builder_current = Builder()
    graph_current = builder_current.build(trees_current)

    diff = parent_commit.diff(current_commit, create_patch = True)
    diff_info = diff_metadata(diff)
    if not diff_info["changed_files"]:
        raise ValueError("Tidak ada perubahan file Python pada commit pair ini")

    try:
        parent_graph = prepare_pruned_graph(
            builder_parent, trees_parent, diff_info["old_ranges"], max_nodes=MAX_NODES
        )
        current_graph = prepare_pruned_graph(
            builder_current, trees_current, diff_info["new_ranges"], max_nodes=MAX_NODES
        )

    except Exception as e:
        print(f"Error occurred while preparing pruned graphs: {e}")
        continue

    change_features = pair_features(diff_info)

    parent_input_tensor = Tensor(
        parent_graph,
        graph_features=change_features,
        local_change_features=change_features
    )

    current_input_tensor = Tensor(
        current_graph,
        graph_features=change_features,
        local_change_features=change_features
    )

    val_data.append({
        "parent": to_data(parent_input_tensor),
        "current": to_data(current_input_tensor),
        "features": torch.from_numpy(change_features).unsqueeze(0),
        "y": torch.tensor([1 if row["is_bug"] else 0], dtype=torch.long),
        "project": project_name,
        "commit_sha": row["commit_sha"],
    })

print(f"Prepared val data for {len(val_data)} commit pairs.")

In [ ]:
test_data = []

for i, row in df_test.iterrows():
    project_name = row["repo"]
    current_commit_hash = row["commit"]

    repo_path = os.path.join(REPO_PATH, project_name)    

    repo = Repo(repo_path)
    current_commit = repo.commit(current_commit_hash)
    parent_commit = repo.head.commit

    if current_commit.hexsha != str(current_commit_hash):
        print(f"commit_sha tidak cocok dengan commit yang dibaca. Skipping...")
        continue

    trees_parent, parent_parse_errors = parse_commit_python_files(parent_commit)
    trees_current, current_parse_errors = parse_commit_python_files(current_commit)

    builder_parent = Builder()
    graph_parent = builder_parent.build(trees_parent)
    
    builder_current = Builder()
    graph_current = builder_current.build(trees_current)

    diff = parent_commit.diff(current_commit, create_patch = True)
    diff_info = diff_metadata(diff)
    if not diff_info["changed_files"]:
        raise ValueError("Tidak ada perubahan file Python pada commit pair ini")

    try:
        parent_graph = prepare_pruned_graph(
            builder_parent, trees_parent, diff_info["old_ranges"], max_nodes=MAX_NODES
        )
        current_graph = prepare_pruned_graph(
            builder_current, trees_current, diff_info["new_ranges"], max_nodes=MAX_NODES
        )

    except Exception as e:
        print(f"Error occurred while preparing pruned graphs: {e}")
        continue

    change_features = pair_features(diff_info)

    parent_input_tensor = Tensor(
        parent_graph,
        graph_features=change_features,
        local_change_features=change_features
    )

    current_input_tensor = Tensor(
        current_graph,
        graph_features=change_features,
        local_change_features=change_features
    )

    test_data.append({
        "parent": to_data(parent_input_tensor),
        "current": to_data(current_input_tensor),
        "features": torch.from_numpy(change_features).unsqueeze(0),
        "y": torch.tensor([1 if row["is_bug"] else 0], dtype=torch.long),
        "project": project_name,
        "commit_sha": row["commit_sha"],
    })

print(f"Prepared test data for {len(test_data)} commit pairs.")

In [ ]:
import random
import copy
from torch.utils.data import DataLoader
from torch_geometric.data import Batch
from sklearn.metrics import f1_score

SEED = 42
BATCH_SIZE = 16
EPOCHS = 100
PATIENCE = None
HIDDEN_CHANNELS = 64
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.0001

np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def collate(items):
    parent = Batch.from_data_list([d["parent"] for d in items])
    current = Batch.from_data_list([d["current"] for d in items])
    feats = torch.log1p(torch.cat([d["features"] for d in items]))
    y = torch.cat([d["y"] for d in items])
    return parent, current, feats, y

train_loader = DataLoader(training_data, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)

counts = torch.bincount(torch.cat([d["y"] for d in training_data]), minlength=2).float()
if (counts == 0).any():
    raise ValueError(f"Train hanya berisi satu kelas {counts.tolist()}. Tambahkan negatif non-bug ke df sebelum training.")
class_weight = (counts.sum() / (2 * counts)).to(device)

model = PairRGCN(
    in_channels=training_data[0]["parent"].x.size(1),
    hidden_channels=HIDDEN_CHANNELS,
    graph_feat_dim=training_data[0]["features"].size(1),
).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

def run_epoch(loader, train=False):
    model.train(train)
    total_loss, n, probs, labels = 0.0, 0, [], []
    with torch.set_grad_enabled(train):
        for parent, current, feats, y in loader:
            parent, current, feats, y = parent.to(device), current.to(device), feats.to(device), y.to(device)
            logits = model(parent, current, feats)
            loss = criterion(logits, y)
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item() * y.size(0)
            n += y.size(0)
            probs.append(F.softmax(logits, dim=-1)[:, 1].detach().cpu())
            labels.append(y.cpu())
    return total_loss / n, torch.cat(probs).numpy(), torch.cat(labels).numpy()

history, best_val, best_state, wait = [], float("inf"), None, 0
for epoch in range(1, EPOCHS + 1):
    train_loss, p_tr, y_tr = run_epoch(train_loader, train=True)
    val_loss, p_va, y_va = run_epoch(val_loader)
    train_f1 = f1_score(y_tr, p_tr >= 0.5, zero_division=0)
    val_f1 = f1_score(y_va, p_va >= 0.5, zero_division=0)
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "train_f1": train_f1, "val_f1": val_f1})
    print(f"epoch {epoch:02d} | loss {train_loss:.4f}/{val_loss:.4f} | f1 {train_f1:.3f}/{val_f1:.3f}")

    if val_loss < best_val - 1e-4:
        best_val, best_state, wait = val_loss, copy.deepcopy(model.state_dict()), 0
    else:
        wait += 1
        if PATIENCE is not None and wait >= PATIENCE:
            print(f"Early stopping di epoch {epoch}")
            break

model.load_state_dict(best_state)
torch.save(best_state, "best_pair_rgcn.pt")
print(f"Val loss terbaik: {best_val:.4f}")

In [ ]:
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, average_precision_score)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

model.load_state_dict(best_state)
_, p_val, y_val = run_epoch(val_loader)
_, p_test, y_test = run_epoch(test_loader)

# threshold dari validasi
thresholds = np.linspace(0.05, 0.95, 91)
if len(set(y_val)) == 2:
    best_t = float(thresholds[int(np.argmax([f1_score(y_val, p_val >= t, zero_division=0) for t in thresholds]))])
else:
    best_t = 0.5
    print("Val hanya satu kelas, threshold dikembalikan ke 0.5")
print(f"Threshold terpilih: {best_t:.2f}\n")

pred = (p_test >= best_t).astype(int)
print(classification_report(y_test, pred, target_names=["likely_bug"], zero_division=0))
print("Confusion matrix [baris=asli, kolom=prediksi]:")
print(confusion_matrix(y_test, pred, labels=[0, 1]))
if len(set(y_test)) == 2:
    print(f"ROC-AUC: {roc_auc_score(y_test, p_test):.4f} | PR-AUC: {average_precision_score(y_test, p_test):.4f}")

# baseline: hanya fitur pasangan, tanpa graph
def feature_matrix(subset):
    return np.log1p(np.concatenate([d["features"].numpy() for d in subset]))

if len(set(y_test)) == 2:
    scaler = StandardScaler().fit(feature_matrix(training_data))
    y_train_np = np.concatenate([d["y"].numpy() for d in training_data])
    baseline = LogisticRegression(class_weight="balanced", max_iter=1000).fit(scaler.transform(feature_matrix(train_set)), y_train_np)
    p_base = baseline.predict_proba(scaler.transform(feature_matrix(test_set)))[:, 1]
    print(f"\nBaseline fitur saja -> ROC-AUC: {roc_auc_score(y_test, p_base):.4f} | PR-AUC: {average_precision_score(y_test, p_base):.4f}")
    print("Kalau mirip dengan model RGCN, graph belum dimanfaatkan.")